<p style="font-size:11px;"><em><strong>Créditos</strong>: El contenido de este cuaderno ha sido tomado de varias fuentes. El compilador se disculpa por cualquier omisión involuntaria y estaría encantado de agregar un reconocimiento.</em></p>

# Modelos con dependencia y hetereogeneidad espacial

Una característica fundamental de los datos espaciales es la presencia simultánea de dependencia espacial y heterogeneidad espacial. La dependencia espacial se refiere al hecho de que las observaciones cercanas tienden a parecerse más entre sí que las distantes, debido a procesos subyacentes que operan en el espacio geográfico. Por su parte, la heterogeneidad espacial se manifiesta en diferencias sistemáticas entre regiones o ubicaciones, ya sea por condiciones ambientales, sociales o geológicas que varían en el territorio. Para abordar estas dos dimensiones, es necesario emplear modelos espaciales que puedan representarlas explícitamente en su estructura.

Los modelos autoregresivos condicionales (CAR), implementados en la librería INLA, permiten incorporar simultáneamente estos dos aspectos en un marco bayesiano. La dependencia espacial se modela mediante una matriz de vecindad (por ejemplo, tipo Queen o Rook), y la heterogeneidad se introduce agregando un efecto aleatorio no estructurado (iid), como en el modelo BYM (Besag-York-Mollié). Por otro lado, en la familia de modelos simultáneos autoregresivos (SAR) —implementados en R mediante la librería spatialreg— la dependencia espacial puede incorporarse a través de la variable dependiente (modelo SARlag), de los errores (SARerror), o de las covariables (modelo SLX). En estos modelos, la heterogeneidad espacial se representa explícitamente mediante interacciones entre las covariables continuas y un factor espacial categórico, como subcuencas o regiones, permitiendo que los efectos de las variables explicativas cambien según la unidad espacial. Este tipo de interacción permite capturar variaciones estructurales (regímenes espaciales) sin necesidad de introducir efectos aleatorios, y es una forma eficaz de modelar heterogeneidad espacial en un contexto frecuentista.

## INLA

Tradicionalmente, los modelos jerárquicos bayesianos han sido una herramienta poderosa para abordar estas características. No obstante, su implementación mediante métodos de muestreo como Markov Chain Monte Carlo (MCMC) puede resultar computacionalmente costosa, especialmente en contextos con grandes volúmenes de datos o estructuras espaciales complejas.

La librería INLA (Integrated Nested Laplace Approximation) para el lenguaje R se presenta como una alternativa eficiente y precisa para ajustar modelos bayesianos latentes gaussianos (LGMs). A través de aproximaciones deterministas, INLA permite estimar la distribución posterior de los parámetros sin recurrir a simulaciones estocásticas, lo que reduce drásticamente los tiempos de cómputo. INLA es especialmente útil para incorporar efectos espaciales estructurados, como los que se modelan mediante procesos autoregresivos condicionales (CAR), y también para representar campos espaciales continuos mediante el enfoque basado en ecuaciones diferenciales estocásticas (SPDE). Esta flexibilidad la convierte en una herramienta ideal para abordar tanto la dependencia espacial estructurada como la variabilidad no estructurada, facilitando la construcción de modelos robustos y adaptados a la complejidad geográfica de los fenómenos estudiados.

A continuación se presenta modelos de regresión de Poisson utilizando la librería INLA, incorporando dos tipos de efectos aleatorios: un intercepto aleatorio para cuenca y un efecto espacial CAR para las subcuencas.

```r
library(sf) # Simple features for spatial data
library(spdep) # Spatial dependence tools
library(spatialreg) # Spatial regression models
library(sf)
library(dplyr)
library(sp)
library(spdep)
library(INLA) 

aoi = st_read("https://github.com/edieraristizabal/ModeloMultinivel/blob/main/DATA/df_catchments_spatial.gpkg")
aoi <- aoi %>% mutate_at(c('elev_mean','slope_mean','RainfallDaysmean'), ~(scale(.) %>% as.vector))
aoi <- aoi %>% mutate(cuenca_num = case_when(cuenca == 'Atrato' ~ 1, cuenca == 'Cauca' ~ 2, cuenca == 'Magdalena' ~ 3))
aoi_sp <- as_Spatial(aoi)

#Spatial matrix
W.nb <- poly2nb(aoi)
W.mat <- nb2mat(W.nb, style="B")
W.list <- nb2listw(W.nb, style="B")


# intercepto aleatorio + ICAR model
m3_icar <- inla(formula = lands_rec ~ 1 + RainfallDaysmean + elev_mean + slope_mean + 
                  f(cuenca, model = "iid") +
                  f(id, model = "besag", graph = aoi.mat),
                  offset=log(area), data = as.data.frame(aoi), family ="poisson",
                  control.predictor = list(compute = TRUE),
                  control.compute = list(dic = TRUE, waic = TRUE))

summary(m3_icar)
m3_icar$summary.random$cuenca
aoi$ICAR <- m3_icar$summary.fitted[1:nrow(aoi), "mean"]
res_pearson_m3 <- (aoi$lands_rec - m3_icar$summary.fitted.values$mean) / sqrt(m3_icar$summary.fitted.values$mean)
aoi$res_pearson_m3 <- res_pearson_m3
moran_m3 <- moran.mc(x = res_pearson_m3, listw = aoi.listw,nsim = 999, alternative = "greater")
```

```r
# BYM model
m4_bym<-inla(formula = lands_rec ~ 1 + RainfallDaysmean + elev_mean + slope_mean +
           f(cuenca, model = "iid") +
           f(id, model = "bym",graph = aoi.mat), 
           offset=log(area), data = as.data.frame(aoi), family = "poisson",
           control.compute = list(dic = TRUE, waic=T), 
           control.predictor = list(compute = TRUE))

summary(m4_bym)
m4_bym$summary.random$cuenca
aoi$BYM <- m4_bym$summary.fitted[1:nrow(aoi), "mean"]
aoi_sp$BYM <- m4_bym$summary.fitted.values[, "mean"]
res_pearson_m4 <- (aoi$lands_rec - m4_bym$summary.fitted.values$mean) / sqrt(m4_bym$summary.fitted.values$mean)
aoi$res_pearson_m4 <- res_pearson_m4
moran_m4 <- moran.mc(x = res_pearson_m4, listw = aoi.listw,nsim = 999, alternative = "greater")
```

```r
#Leroux et al. model
m5_leroux <- Diagonal(nrow(aoi.mat), apply(aoi.mat, 1, sum)) - aoi.mat
Cmatrix <- Diagonal(nrow(aoi), 1) -  m5_leroux
max(eigen(Cmatrix)$values) #just to check =1

m5_ler = inla(formula = lands_rec ~ 1 + RainfallDaysmean + elev_mean + slope_mean +
                f(cuenca, elev_mean, model = "iid") +
                f(id, model = "generic1", Cmatrix = Cmatrix),
                offset=log(area), data = as.data.frame(aoi), family ="poisson",
                control.predictor = list(compute = TRUE),
                control.compute = list(dic = TRUE, waic = TRUE))

summary(m5_ler)
m5_ler$summary.random$cuenca
aoi$LEROUX <- m5_ler$summary.fitted[1:nrow(aoi), "mean"]
res_pearson_m5 <- (aoi$lands_rec - m5_ler$summary.fitted.values$mean) / sqrt(m5_ler$summary.fitted.values$mean)
aoi$res_pearson_m5 <- res_pearson_m5
moran_m5 <- moran.mc(x = res_pearson_m5, listw = aoi.listw,nsim = 999, alternative = "greater")
```

Para añadir coeficientes aleatorios espaciales (p. ej. para la pendiente) en un modelo CAR con INLA, lo más sencillo es definir un segundo campo latente que modere justamente ese covariable. La idea es que en la fórmula incluyas: (i) Un campo latente para el intercepto espacial (como ya haces con f(id, model="besag", graph=graph)), y (ii) Un campo latente “pesado” por la covariable cuya pendiente quieres aleatorizar (por ejemplo slope_mean).

```r
# Primero, crea dos índices idénticos (uno para intercepto y otro para pendiente)
aoi$id_int   <- 1:nrow(aoi)
aoi$id_slope <- aoi$id_int

# Ahora la fórmula:
m6_spatial_slope <- inla(
  lands_rec ~
    1
    + RainfallDaysmean
    + elev_mean
    + slope_mean                  # efecto fijo de la pendiente
    + f(cuenca, model="iid")      # efecto aleatorio no espacial por cuenca
    + f(id_int,   model="besag", graph=cl_graph)              # intercepto espacial
    + f(id_slope, slope_mean,      model="besag", graph=cl_graph),  # pendiente espacial
  offset = log(area),
  family = "poisson",
  data = aoi,
  control.predictor = list(compute = TRUE),
  control.compute   = list(dic = TRUE, waic = TRUE)
)
```


Si tu objetivo fuera que la pendiente de slope_mean variara entre cuencas (no dentro de ellas), basta con usar el índice cuenca en lugar de id_slope. Por ejemplo:

```r
m7_basin_slope <- inla(
  lands_rec ~
    1 +
    RainfallDaysmean +
    elev_mean +
    slope_mean +
    f(cuenca,   model="iid") +                         # intercepto por cuenca
    f(cuenca, slope_mean, model="iid"),                # pendiente aleatoria por cuenca
  offset = log(area),
  family = "poisson",
  data = aoi,
  control.predictor = list(compute=TRUE),
  control.compute   = list(dic=TRUE, waic=TRUE)
)
```

Para incluir a la vez: (i) Intercepto aleatorio por cuenca, (ii) Pendientes aleatorias por cuenca para cada covariable, y (iii) Un efecto CAR espacial a nivel de ID (cada polígono). puedes usar una fórmula como esta en R‑INLA:

```r
# índice numérico de cuenca (1,2,3)  
aoi$basin_id <- as.integer(aoi$cuenca)  
# índice numérico de polígono (ID)  
aoi$id        <- 1:nrow(aoi)
```

```r
m_complex <- inla(
  lands_rec ~
    1
    + RainfallDaysmean          # efecto fijo
    + elev_mean                 # efecto fijo
    + slope_mean                # efecto fijo

    # 1) Intercepto aleatorio por cuenca
    + f(basin_id, model="iid")

    # 2) Pendientes aleatorias por cuenca
    + f(basin_id, RainfallDaysmean, model="iid")
    + f(basin_id, elev_mean,        model="iid")
    + f(basin_id, slope_mean,       model="iid")

    # 3) Efecto espacial CAR a nivel de polígono
    + f(id, model="besag", graph=cl_graph),

  data = aoi,
  family = "poisson",
  offset = log(area),
  control.predictor = list(compute = TRUE),
  control.compute   = list(dic = TRUE, waic = TRUE)
)
```


## *Spatial reg*

A continuación se presenta un modelo SAR que está intentando explicar la variación en la variable dependiente transformada (y_transf) en función de la elevación media, la pendiente media y la precipitación media anual, permitiendo que la relación entre estas variables predictoras y la variable dependiente sea diferente para cada cuenca. Además, el componente SAR del modelo tiene en cuenta la autocorrelación espacial en la variable dependiente, lo que significa que el valor de y_transf en una ubicación está influenciado por los valores de y_transf en sus ubicaciones vecinas (definidas por aoi.listw). La ausencia de un intercepto implica que el modelo está restringido a pasar por el origen para cada cuenca en el espacio de las variables predictoras.

```r
# Regímenes - Modelo de Retardo Espacial (SAR) - cuenca
col.fit9 <- lagsarlm(y_transf ~ 0 + (elev_mean + slope_mean + RainfallDaysmean):(cuenca), data = aoi2,listw = aoi.listw)
summary(col.fit9,Nagelkerke=T)

# Regímenes - Modelo de Error Espacial (SEM) - cuenca
col.fit11 = errorsarlm(y_transf ~ 0 + (elev_mean + slope_mean + RainfallDaysmean):(cuenca), data = aoi2,listw = aoi.listw)
summary(col.fit11,Nagelkerke=T)

# Regímenes - Modelo Mansky (SARAR con restricciones) - cuenca
col.fit14 <- sacsarlm(y_transf ~ 0 + (elev_mean + slope_mean + RainfallDaysmean):(cuenca), data = aoi2,listw = aoi.listw, type="sacmixed")
summary(col.fit14,Nagelkerke=T)

# Regímenes - Modelo SAC (Simultaneous Autoregressive and Conditional Autoregressive) - cuenca
col.fit15 <- sacsarlm(y_transf ~ 0 + (elev_mean + slope_mean + RainfallDaysmean):(cuenca), data = aoi2,listw = aoi.listw, type="sac")
summary(col.fit15)

# Regímenes - Modelo SLX (Spatial Lag of X) - cuenca
col.fit17 = lmSLX(y_transf ~ 0 + (elev_mean + slope_mean + RainfallDaysmean):(cuenca), data = aoi2,listw = aoi.listw)
summary(col.fit17,Nagelkerke=T)
AIC(col.fit17)

# Regímenes - Modelo SDM (Spatial Durbin Model) - cuenca
col.fit19 = lagsarlm(y_transf ~ 0 + (elev_mean + slope_mean + RainfallDaysmean):(cuenca), data = aoi2,listw = aoi.listw,type = "mixed")
summary(col.fit19,Nagelkerke=T)
```